# Limpieza de datos: Libere - Reto06

**CARGA DE LIBRERIAS**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import missingno as msno
import os
import datetime as datetime

**DESCARGA DE DATOS**

In [ ]:
path = os.path.join('Datos', 'Originales', 'cancellation_data_for_mondragon_unibertsitatea_2024.csv')
df = pd.read_csv(path)

**Analizar Datos**

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

**1) Conversión de Fechas a tipo Fecha**

In [ ]:
#Convertir a tipo fecha, las variables de fechas
formato = "%b %d, %Y, %H:%M"

df["booked_at"] = pd.to_datetime(df["booked_at"], format=formato, errors="coerce")
df["checkin_time"] = pd.to_datetime(df["checkin_time"], format=formato, errors="coerce")
df["checkout_time"] = pd.to_datetime(df["checkout_time"], format=formato, errors="coerce")
df["asset_opening_date"] = pd.to_datetime(df["asset_opening_date"], format=formato, errors="coerce")
df["last_entry_form_completed_at"] = pd.to_datetime(df["last_entry_form_completed_at"], format = formato, errors="coerce")
df["cancelled_at"] = pd.to_datetime(df["cancelled_at"], format = formato, errors="coerce")

#Convertir a numerico:
columnas_numericas = ['lenght_of_stay','adult_count','child_count','completed_entry_forms_count','product_count','total_adr','cancellation_lead_time']

for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df.dtypes

**2) Analizar DUPLICADOS**
#### Los eliminaremos antes de hacer las imputaciones

In [ ]:
df.duplicated().sum()# Numero de duplicados

In [ ]:
print(df)

In [ ]:
#Filas duplicadas
df[df.duplicated()]

In [ ]:
#Quedarnos con el df limpio sin duplicados
df= df.drop_duplicates()
print(df.shape)

**3) Analizar MISSINGS (Valores NA)**

In [ ]:
df.isna()

In [ ]:
#Numero de NA-s  por cada columna
tabla_nas_columnas = df.isna().sum(axis = 0).reset_index(name="N_Nas").sort_values("N_Nas",ascending=False)
tabla_nas_columnas = tabla_nas_columnas[tabla_nas_columnas["N_Nas"]>0]
tabla_nas_columnas

In [ ]:
#Numero de NA-s por cada fila
df.isna().sum(axis = 1).reset_index(name="N_Nas").sort_values("N_Nas",ascending=False)

In [ ]:
#Visualizar la distribucion de los NA-s
msno.matrix(df)

In [ ]:
msno.bar(df) # Instancias completas

3.1) Correlación entre missings

In [ ]:
msno.heatmap(df) 

**4) Tratamientos de MISSINGS**

### -------   IMPUTACIONES  de las Variables con NA-s

##### 4.1) Imputacion Variables: `Cancellation_reason, Cancellation_lead_time, Cancelled_at`
##### (Las hemos imputado con "not_cancelled" o como una fecha neutral (21990-01-01))

In [ ]:
#Rellenar los NA de estas columnas con "not_cancelled" o "0"
df["cancellation_reason"].fillna("not_cancelled", inplace=True)
df["cancellation_lead_time"].fillna(pd.Timestamp("1900-01-01"), inplace=True)
df["cancelled_at"].fillna(pd.Timestamp("1900-01-01"), inplace=True)


##### 4.2) Imputacion Variable: `travel_agency`
###### Tras analizar los NA de esta varibales se ha concluido que es por la razon de que la reserva se ha realizado, en persona,email,telefono..


In [ ]:
df["travel_agency_name"].fillna("agency_no_registrada", inplace=True)


##### 4.3) Imputacion Variable: `completed_entry_forms_count`
###### Se imputó según `all_entry_forms_completed`. Si esta era "no", significa que no se ha completado ningún informe, por lo que asignamos 0.Ademas se ha actualizado "all_entry_forms_completed" a "yes" si hay algún informe completado



In [ ]:
#Imputar los valores NA-s:
df.loc[df["all_entry_forms_completed"] == "no", "completed_entry_forms_count"] = 0

In [ ]:
#Identificar los que "all_entry_forms_completed" son "no" pero han realizado algun test, es decir, que "completed_entry_forms_count" != 0
df.loc[df["completed_entry_forms_count"] > 0, "all_entry_forms_completed"] = "yes"

#Comprobar que se ha realizado correctamente:
cantidad_no_no0 = df[(df["completed_entry_forms_count"] > 0) & (df["all_entry_forms_completed"] == "no")].shape[0]
print(f'Quedan {cantidad_no_no0} de casos en el que "completed_entry_forms" es mayor que 0 y "all_entry_forms" y es igual a "no". Por lo que esta correctamnete hecho!')

##### 4.5) Imputacion Variable : `last_entry_form_completed_at`
###### Imputaremos los valores NA-s de esta variable con una fecha “neutral” en este caso ("1900-01-01")

In [ ]:
df["last_entry_form_completed_at"].fillna(pd.Timestamp("1900-01-01"), inplace=True)

In [ ]:
#NUEVA COLUMNA DE RATIO DE FORMULARIOS COMPLETADOS POR PERSONAS EN LA RESERVA: NUMERO DE FORMULARIOS / NUMERO DE PERSONAS QUE ACUDIERON AL HOTEL (ADULT COUNT+ CHILD)

##### 4.6) Imputacion Variable: `total_adr`
###### Solo hay un NA-s y se ha imputado como 0 debido a que la variable "reservation_net_value" era 0

In [ ]:
#Hay un NA en esta variable y es devido a que la variable "reservation_net_value" es 0. POr loq eu esta tmabien se imputara como 0
df["total_adr"].fillna(0, inplace=True)

In [ ]:
#Comprobacion de si siguen quedando NA-s
#Las columnas con más NAs
tabla_nas_columnas1 =  df.isna().sum()#Muestra cuantas columnas tienen NA
tabla_nas_columnas1 [tabla_nas_columnas1>0]

################################################################################################

##### Creacion nueva columna (Localicacion Hotel Reserva)

In [ ]:
mapeo_ciudades = {
    'Koisi Hostel': 'Donostia',
    'Líbere Vitoria': 'Vitoria',
    'Líbere Bilbao Museo': 'Bilbao',
    'Líbere Bilbao La Vieja': 'Bilbao',
    'Líbere Valencia Abastos': 'Valencia',
    'Líbere Valencia Jardín Botánico': 'Valencia',
    'Líbere Madrid Palacio Real': 'Madrid',
    'Líbere Málaga Teatro Romano': 'Málaga',
    'Líbere Granada Catedral': 'Granada',
    'Líbere Málaga la Merced': 'Málaga',
    'Líbere Córdoba Patio Santa Marta': 'Córdoba',
    'Líbere Pamplona Yamaguchi': 'Pamplona'
}
df['City'] = df['asset'].map(mapeo_ciudades)

In [ ]:
df.duplicated().sum()

# LIMPIEZA MAS EXTENSA

##### Creacion Columna : Dia de la Semana

In [ ]:
df["Reserva_Dia_Semana"] = df["booked_at"].dt.day_name()
df["Cancelacion_Dia_Semana"] = df["cancelled_at"].dt.day_name()

##### Creacion Columna: Estacion (Reserva / Estancia / Cancelacion)

In [ ]:
def mes_a_estacion(mes):
    if mes in [12, 1, 2]:
        return 'Winter'
    elif mes in [3, 4, 5]:
        return 'Spring'
    elif mes in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df['Estacion_Cancelacion'] = df['cancelled_at'].dt.month.apply(mes_a_estacion)
df['Estacion_Estancia'] = df['checkin_time'].dt.month.apply(mes_a_estacion)
df['Estacion_Reserva'] = df['booked_at'].dt.month.apply(mes_a_estacion)

In [ ]:
print(df.head(10))

##### Analizar zona roja de cancelaciones
*Zona Roja Mensual* : Agosto, Mayo, Junio      //////
*Zona Roja Estacional* : Primavera

Se han creado dos variables nuevas; si la cancelacion se ha realizado en uno de esos meses o estacion el valor es 1, en caso contrario 0.

### Cálculo del ratio de cancelación por Mes / Estacion

Este bloque calcula la **proporción de cancelaciones por mes**.  
- Primero, se agrupa las reservas por `checkin_month`.  
- Luego se cuenta:
  - `Canceladas`: número de reservas con `status = 'cancelled'`.  
  - `Total_Reservas`: número total de reservas en ese mes.  
- A continuación, se calcula el **Ratio_Cancelacion** como `Canceladas / Total_Reservas`.  
- Finalmente, se ordena de mayor a menor para identificar los meses con mayor riesgo de cancelación, que se usarán para definir la **zona roja de cancelaciones**.


In [ ]:
#-----Cálculo de ratio de cancelación por mes
# Cancelaciones / Total reservas por mes
cancel_ratio_month = (
    df.groupby("checkin_month")
    .agg(
        Canceladas=("status", lambda x: (x == "cancelled").sum()),
        Total_Reservas=("status", "count")
    )
    .assign(Ratio_Cancelacion=lambda x: x["Canceladas"] / x["Total_Reservas"])
    .sort_values("Ratio_Cancelacion", ascending=False)
)

print("\nRatio de cancelación por mes:")
print(cancel_ratio_month)


In [ ]:
#-----Cálculo de ratio de cancelación por estacion
cancel_ratio_season = (
    df.groupby("Estacion_Cancelacion")
    .agg(
        Canceladas=("status", lambda x: (x == "cancelled").sum()),
        Total_Reservas=("status", "count")
    )
    .assign(Ratio_Cancelacion=lambda x: x.Canceladas / x.Total_Reservas)
    .sort_values("Ratio_Cancelacion", ascending=False)
)

print("Ratio de cancelación por estación:")
print(cancel_ratio_season)



#### Zona Roja: 
Este bloque identifica los **meses y estaciones con mayor riesgo de cancelación**:

1. **Zona roja mensual**:  
   - Calcula la media del `Ratio_Cancelacion` por mes (`media_ratio_mensual`).  
   - Los meses cuyo ratio de cancelación es **mayor que la media** se consideran **zona roja** y se almacenan en `meses_zona_roja`.

2. **Zona roja estacional**:  
   - Calcula la media del `Ratio_Cancelacion` por estación (`media_ratio_estacional`).  
   - Las estaciones con ratio superior a la media se consideran **zona roja estacional** y se guardan en `estaciones_zona_roja`.

Estas zonas rojas servirán para marcar reservas con mayor probabilidad de cancelación en análisis y modelos predictivos.


In [ ]:
#Definir zona rojas MENSUAL:
media_ratio_mensual = cancel_ratio_month["Ratio_Cancelacion"].mean()
print(media_ratio_mensual)
# Meses zona roja (ratio > media)
meses_zona_roja = cancel_ratio_month[cancel_ratio_month["Ratio_Cancelacion"] > media_ratio_mensual].index.tolist()

print("MESES ZONA ROJA (ratio > media):")
print(meses_zona_roja)

#----------------------------------------------------
print( "--------------------------------------------------")
# Definir zona roja ESTACIONAL
media_ratio_estacional = cancel_ratio_season["Ratio_Cancelacion"].mean()
estaciones_zona_roja = cancel_ratio_season[cancel_ratio_season["Ratio_Cancelacion"] > media_ratio_estacional].index.tolist()

print("ESTACIONES ZONA ROJA (ratio > media):")
print(estaciones_zona_roja)

##### Creacion Columna : Zona Roja Mensual/ Estacional
Se crean dos columnas para marcar si la reserva está en **mes o estación de alta cancelación**:  
- `zona_roja_mes` = 1 si el periodo de reserva coincide con un mes de riesgo.  
- `zona_roja_estacion` = 1 si la reserva o estancia coincide con una estación de riesgo.  
NA si falta información. Esto ayuda a identificar clientes con mayor probabilidad de cancelar.


In [ ]:
# ---------- FUNCIÓN PARA ZONA ROJA MENSUAL ----------
def ventana_zona_roja_mes(row):
    if pd.isna(row['booked_at']) or pd.isna(row['checkin_time']):
        return np.nan   # <<--- ahora en lugar de 0 devolvemos NA
    
    meses_entre = pd.date_range(start=row['booked_at'], end=row['checkin_time'], freq='MS') \
                        .month_name().tolist()
    
    return 1 if any(mes in meses_zona_roja for mes in meses_entre) else 0



# ---------- FUNCIÓN PARA ZONA ROJA ESTACIONAL ----------
def zona_roja_estacion(row):
    if pd.isna(row['Estacion_Cancelacion']) or pd.isna(row['Estacion_Reserva']):
        return np.nan   # <<--- ahora igual que en mensual
    
    return 1 if (
        row['Estacion_Cancelacion'] in estaciones_zona_roja
        or row['Estacion_Reserva'] in estaciones_zona_roja
    ) else 0

# Crear las columnas
df['zona_roja_mes'] = df.apply(ventana_zona_roja_mes, axis=1)
df['zona_roja_estacion'] = df.apply(zona_roja_estacion, axis=1)

##### Creacion Columna: CheckIn (Antes, Durante, Despues)

### Clasificación del momento relativo a la estancia

Se crea la columna `momento_checkin` para indicar **cuándo se completó el último formulario** relativo a la estancia:

- **Antes**: el formulario se completó **antes del check-in**.  
- **Durante**: el formulario se completó **entre el check-in y el checkout**.  
- **Después**: el formulario se completó **después del checkout**.  
- **NA**: si no hay registro del formulario (`last_entry_form_completed_at` es nulo).

Esta variable ayuda a medir el **compromiso del cliente** y puede relacionarse con la probabilidad de cancelación.


In [ ]:
# Clasificar el momento relativo a la estancia
def momento_checkin(row):
    if pd.isna(row['last_entry_form_completed_at']):
        return None  # No hay evento registrado
    elif row['last_entry_form_completed_at'] < row['checkin_time']:
        return 'Antes'
    elif row['checkin_time'] <= row['last_entry_form_completed_at'] <= row['checkout_time']:
        return 'Durante'
    else:  # last_entry_form_completed_at > checkout_time
        return 'Después'

# Crear nueva columna
# -----------------------------
df['momento_checkin'] = df.apply(momento_checkin, axis=1)

##### Creacion Columna: Ratio Formularios= (Completed entry forms count / adult count + child count)

- `1` → Se completó un formulario por cada persona de la reserva.

- `<1` → Faltan formularios para algunas personas.

- `>1` → Hay formularios extra (si alguien completó más de uno por persona).

In [ ]:
# Número total de personas por reserva
df['total_personas_adult_y_niños'] = df['adult_count'] + df['child_count']

# Calcular ratio de formularios completados
# -----------------------------
df['ratio_formularios'] = df['completed_entry_forms_count'] / df['total_personas_adult_y_niños']


### Creacion Columna: reservation_checkin_month_diff
- Diferencia en meses entre el mes de la reserva (booked_at) y el mes del check-in (checkin_time).
- Reservas hechas con mucha antelación pueden cancelar más por cambios de planes, o menos si son corporativas.

In [ ]:
df['reservation_checkin_month_diff'] = (df['checkin_time'].dt.year - df['booked_at'].dt.year) * 12 + \
                               (df['checkin_time'].dt.month - df['booked_at'].dt.month)


### Creacion Columna: estancia_en_festivo
Detectar si durante la estancia del huésped (desde el check-in y durante toda la duración de la reserva) hay algún día festivo en España.
- `1` = La reserva toca o atraviesa un festivo.
- `0` = La reserba no toca o no atraviersa un dia festivo <br>
Nos permite captura comportamientos de reserva asociados a festivos: los clientes pueden reservar más apresuradamente o cambiar planes.

In [ ]:
from datetime import datetime

festivos = [
    # 2022
    datetime(2022, 1, 1),   # Año Nuevo
    datetime(2022, 1, 6),   # Reyes Magos
    datetime(2022, 4, 14),  # Jueves Santo
    datetime(2022, 4, 15),  # Viernes Santo
    datetime(2022, 5, 1),   # Día del Trabajador
    datetime(2022, 6, 16),  # Corpus Christi (móvil, ejemplo para algunas regiones)
    datetime(2022, 8, 15),  # Asunción
    datetime(2022, 10, 12), # Fiesta Nacional
    datetime(2022, 11, 1),  # Todos los Santos
    datetime(2022, 12, 6),  # Día de la Constitución
    datetime(2022, 12, 8),  # Inmaculada Concepción
    datetime(2022, 12, 25), # Navidad

    # 2023
    datetime(2023, 1, 1),   # Año Nuevo
    datetime(2023, 1, 6),   # Reyes Magos
    datetime(2023, 4, 6),   # Jueves Santo
    datetime(2023, 4, 7),   # Viernes Santo
    datetime(2023, 5, 1),   # Día del Trabajador
    datetime(2023, 6, 8),   # Corpus Christi (móvil)
    datetime(2023, 8, 15),  # Asunción
    datetime(2023, 10, 12), # Fiesta Nacional
    datetime(2023, 11, 1),  # Todos los Santos
    datetime(2023, 12, 6),  # Día de la Constitución
    datetime(2023, 12, 8),  # Inmaculada Concepción
    datetime(2023, 12, 25)  # Navidad
]


In [ ]:
import holidays

festivos_espana = holidays.Spain(years=2023)

# Generar rango de días completos de la estancia
def estancia_toca_festivo(row):
    fechas_estancia = pd.date_range(start=row['checkin_time'],
                                    periods=row['lenght_of_stay'],
                                    freq='D')
    return int(any(f.date() in festivos_espana for f in fechas_estancia))

df['estancia_en_festivo'] = df.apply(estancia_toca_festivo, axis=1)


In [ ]:
#Comprobar si hay muchas reservas duranet fechas de festivo de España:
reservas_festivos_SI = df[df["estancia_en_festivo"] == 1]
reservas_festivos_SI.shape[0]

### Creación Columna: estancia_en_finde
Detectar si durante la estancia del huésped (desde el check-in y durante toda la duración de la reserva) hay algún día de fin de semana (sábado o domingo).  
- `1` = La estancia incluye al menos un día de fin de semana.  
- `0` = La estancia no incluye ningún día de fin de semana. <br>
Nos permite capturar comportamientos de reserva asociados a fines de semana: por ejemplo, variaciones en cancelaciones, cambios de planes o patrones de ocupación distintos en días de ocio.


In [ ]:
def estancia_toca_finde(row):
    fechas = pd.date_range(start=row['checkin_time'], periods=row['lenght_of_stay'], freq='D')
    return int(any(f.weekday() >= 5 for f in fechas))  # 5=sábado, 6=domingo

df['estancia_en_finde'] = df.apply(estancia_toca_finde, axis=1)
